In [4]:
# HW08-09 — PyTorch MLP: регуляризация и оптимизация обучения (S08–S09)
# Датасет по умолчанию: KMNIST (вариант A)
# Часть A: E1–E4 (Dropout / BatchNorm / EarlyStopping)
# Часть B: O1–O3 (плохие LR + SGD+momentum + weight decay)
# Артефакты сохраняются в homeworks/HW08-09/artifacts/

import os
import json
import csv
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Dict, Any, Tuple

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split

import torchvision
from torchvision import transforms


# -----------------------------
# 1) Seed + device
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


# -----------------------------
# 2) Paths (VS Code friendly)
# -----------------------------
def find_repo_root(start: Path) -> Path:
    p = start.resolve()
    for _ in range(10):
        if (p / "homeworks").exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    return start.resolve()

REPO_ROOT = find_repo_root(Path.cwd())

HW_DIR = REPO_ROOT / "homeworks" / "HW08-09"
ART_DIR = HW_DIR / "artifacts"
FIG_DIR = ART_DIR / "figures"
DATA_DIR = REPO_ROOT / "data"

HW_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("HW_DIR:", HW_DIR)
print("ART_DIR:", ART_DIR)
print("FIG_DIR:", FIG_DIR)
print("DATA_DIR:", DATA_DIR)


# -----------------------------
# 3) Dataset + transforms
# -----------------------------
# Choose one: "KMNIST" (A), "EMNIST" (B), "CIFAR10" (C)
DATASET_NAME = "CIFAR10"

if DATASET_NAME in ["KMNIST", "EMNIST"]:
    tfm = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
elif DATASET_NAME == "CIFAR10":
    tfm = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])
else:
    raise ValueError(f"Unknown dataset: {DATASET_NAME}")

def load_datasets(name: str, data_dir: Path, transform):
    if name == "KMNIST":
        train_full = torchvision.datasets.KMNIST(root=str(data_dir), train=True, download=True, transform=transform)
        test_ds = torchvision.datasets.KMNIST(root=str(data_dir), train=False, download=True, transform=transform)
    elif name == "EMNIST":
        train_full = torchvision.datasets.EMNIST(root=str(data_dir), split="balanced", train=True, download=True, transform=transform)
        test_ds = torchvision.datasets.EMNIST(root=str(data_dir), split="balanced", train=False, download=True, transform=transform)
    elif name == "CIFAR10":
        train_full = torchvision.datasets.CIFAR10(root=str(data_dir), train=True, download=True, transform=transform)
        test_ds = torchvision.datasets.CIFAR10(root=str(data_dir), train=False, download=True, transform=transform)
    else:
        raise ValueError(name)
    return train_full, test_ds

train_full, test_ds = load_datasets(DATASET_NAME, DATA_DIR, tfm)

# train/val split (fixed seed)
val_ratio = 0.2
n_total = len(train_full)
n_val = int(n_total * val_ratio)
n_train = n_total - n_val

g = torch.Generator().manual_seed(SEED)
train_ds, val_ds = random_split(train_full, [n_train, n_val], generator=g)

print("Sizes:", len(train_ds), len(val_ds), len(test_ds))


# -----------------------------
# 4) DataLoaders + sanity check
# -----------------------------
BATCH_SIZE = 128
NUM_WORKERS = 0  # safe for VS Code/Windows

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))

x, y = next(iter(train_loader))
print("Sanity batch x:", x.shape, x.dtype, "min/max:", float(x.min()), float(x.max()))
print("Sanity batch y:", y.shape, y.dtype, "min/max:", int(y.min()), int(y.max()))

# infer input_dim / num_classes robustly
sample_x, sample_y = next(iter(train_loader))
input_dim = int(np.prod(sample_x.shape[1:]))

# Prefer dataset targets if available
num_classes = None
if hasattr(train_full, "targets"):
    t = train_full.targets
    if isinstance(t, list):
        t = torch.tensor(t)
    num_classes = int(torch.max(t).item() + 1)
if num_classes is None:
    num_classes = int(torch.max(sample_y).item() + 1)

print("input_dim:", input_dim, "num_classes:", num_classes)


# -----------------------------
# 5) Model
# -----------------------------
class MLP(nn.Module):
    def __init__(self, input_dim: int, num_classes: int, hidden_sizes: Tuple[int, ...],
                 dropout_p: float = 0.0, use_batchnorm: bool = False):
        super().__init__()
        layers = [nn.Flatten()]
        in_f = input_dim

        for h in hidden_sizes:
            layers.append(nn.Linear(in_f, h))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout_p and dropout_p > 0:
                layers.append(nn.Dropout(p=dropout_p))
            in_f = h

        layers.append(nn.Linear(in_f, num_classes))  # logits
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

def model_summary_str(hidden_sizes, dropout_p, use_batchnorm):
    return f"hidden={list(hidden_sizes)}, act=ReLU, dropout={dropout_p}, batchnorm={use_batchnorm}"


# -----------------------------
# 6) Train / eval
# -----------------------------
def accuracy_from_logits(logits: torch.Tensor, y: torch.Tensor) -> float:
    preds = torch.argmax(logits, dim=1)
    return (preds == y).float().mean().item()

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, total_acc, n_batches = 0.0, 0.0, 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits, y)
        n_batches += 1

    return total_loss / n_batches, total_acc / n_batches

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_acc, n_batches = 0.0, 0.0, 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits, y)
        n_batches += 1

    return total_loss / n_batches, total_acc / n_batches


# -----------------------------
# 7) EarlyStopping
# -----------------------------
class EarlyStopping:
    def __init__(self, patience=4, mode="max", min_delta=0.0):
        self.patience = patience
        self.mode = mode
        self.min_delta = min_delta
        self.best = None
        self.bad_epochs = 0
        self.best_state = None

    def _is_improvement(self, value: float) -> bool:
        if self.best is None:
            return True
        if self.mode == "max":
            return value > self.best + self.min_delta
        return value < self.best - self.min_delta

    def step(self, value: float, model: nn.Module) -> bool:
        # returns True if should stop
        if self._is_improvement(value):
            self.best = value
            self.bad_epochs = 0
            self.best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            self.bad_epochs += 1
        return self.bad_epochs >= self.patience


# -----------------------------
# 8) Experiment runner + artifacts helpers
# -----------------------------
@dataclass
class ExperimentResult:
    experiment_id: str
    best_val_acc: float
    best_val_loss: float
    epochs_trained: int
    history: Dict[str, list]
    best_state: Optional[Dict[str, torch.Tensor]]

def make_optimizer(name: str, params, lr: float, momentum: float = 0.0, weight_decay: float = 0.0):
    if name == "Adam":
        return torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)
    if name == "SGD":
        return torch.optim.SGD(params, lr=lr, momentum=momentum, weight_decay=weight_decay)
    raise ValueError(name)

def run_experiment(
    experiment_id: str,
    hidden_sizes: Tuple[int, ...],
    dropout_p: float,
    use_batchnorm: bool,
    optimizer_name: str,
    lr: float,
    momentum: float,
    weight_decay: float,
    max_epochs: int,
    early_stopping: bool = False,
    es_patience: int = 4,
) -> ExperimentResult:
    model = MLP(input_dim=input_dim, num_classes=num_classes, hidden_sizes=hidden_sizes,
                dropout_p=dropout_p, use_batchnorm=use_batchnorm).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = make_optimizer(optimizer_name, model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    best_val_acc = -1.0
    best_val_loss = float("inf")
    best_state = None

    stopper = EarlyStopping(patience=es_patience, mode="max") if early_stopping else None

    for epoch in range(1, max_epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        va_loss, va_acc = evaluate(model, val_loader, criterion, device)

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)

        # best-by-accuracy tracking
        if va_acc > best_val_acc:
            best_val_acc = va_acc
            best_val_loss = va_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if stopper is not None:
            should_stop = stopper.step(va_acc, model)
            if should_stop:
                best_state = stopper.best_state
                best_val_acc = float(stopper.best)
                best_val_loss = float(np.min(history["val_loss"]))
                return ExperimentResult(experiment_id, float(best_val_acc), float(best_val_loss), epoch, history, best_state)

    return ExperimentResult(experiment_id, float(best_val_acc), float(best_val_loss), max_epochs, history, best_state)

def plot_curves(history: Dict[str, list], title: str, out_path: Path):
    epochs = np.arange(1, len(history["train_loss"]) + 1)
    fig, ax = plt.subplots(1, 1, figsize=(9, 5))
    ax.plot(epochs, history["train_loss"], label="train_loss")
    ax.plot(epochs, history["val_loss"], label="val_loss")
    ax.set_xlabel("epoch")
    ax.set_ylabel("loss")
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)

def plot_lr_extremes(hist_big: Dict[str, list], hist_small: Dict[str, list], out_path: Path):
    e1 = np.arange(1, len(hist_big["train_loss"]) + 1)
    e2 = np.arange(1, len(hist_small["train_loss"]) + 1)

    fig, ax = plt.subplots(1, 1, figsize=(9, 5))
    ax.plot(e1, hist_big["train_loss"], label="O1 train_loss (lr too big)")
    ax.plot(e1, hist_big["val_loss"], label="O1 val_loss (lr too big)")
    ax.plot(e2, hist_small["train_loss"], label="O2 train_loss (lr too small)")
    ax.plot(e2, hist_small["val_loss"], label="O2 val_loss (lr too small)")
    ax.set_xlabel("epoch")
    ax.set_ylabel("loss")
    ax.set_title("LR extremes: too big vs too small")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)

RUNS_CSV_PATH = ART_DIR / "runs.csv"
RUNS_FIELDS = [
    "experiment_id", "dataset", "seed", "model_summary", "optimizer", "lr", "momentum", "weight_decay",
    "epochs_trained", "best_val_accuracy", "best_val_loss"
]

def write_runs_csv(rows: list, path: Path):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=RUNS_FIELDS)
        w.writeheader()
        for r in rows:
            w.writerow(r)


# -----------------------------
# 9) Part A (E1-E4)
# -----------------------------
runs_rows = []

HIDDEN = (512, 256)
BASE_LR = 1e-3
MAX_EPOCHS_FIXED = 20

# E1: base
E1 = run_experiment(
    experiment_id="E1",
    hidden_sizes=HIDDEN,
    dropout_p=0.0,
    use_batchnorm=False,
    optimizer_name="Adam",
    lr=BASE_LR,
    momentum=0.0,
    weight_decay=0.0,
    max_epochs=MAX_EPOCHS_FIXED,
    early_stopping=False
)
runs_rows.append({
    "experiment_id": "E1",
    "dataset": DATASET_NAME,
    "seed": SEED,
    "model_summary": model_summary_str(HIDDEN, 0.0, False),
    "optimizer": "Adam",
    "lr": BASE_LR,
    "momentum": 0.0,
    "weight_decay": 0.0,
    "epochs_trained": E1.epochs_trained,
    "best_val_accuracy": E1.best_val_acc,
    "best_val_loss": E1.best_val_loss
})

# E2: Dropout
DROPOUT_P = 0.3
E2 = run_experiment(
    experiment_id="E2",
    hidden_sizes=HIDDEN,
    dropout_p=DROPOUT_P,
    use_batchnorm=False,
    optimizer_name="Adam",
    lr=BASE_LR,
    momentum=0.0,
    weight_decay=0.0,
    max_epochs=MAX_EPOCHS_FIXED,
    early_stopping=False
)
runs_rows.append({
    "experiment_id": "E2",
    "dataset": DATASET_NAME,
    "seed": SEED,
    "model_summary": model_summary_str(HIDDEN, DROPOUT_P, False),
    "optimizer": "Adam",
    "lr": BASE_LR,
    "momentum": 0.0,
    "weight_decay": 0.0,
    "epochs_trained": E2.epochs_trained,
    "best_val_accuracy": E2.best_val_acc,
    "best_val_loss": E2.best_val_loss
})

# E3: BatchNorm
E3 = run_experiment(
    experiment_id="E3",
    hidden_sizes=HIDDEN,
    dropout_p=0.0,
    use_batchnorm=True,
    optimizer_name="Adam",
    lr=BASE_LR,
    momentum=0.0,
    weight_decay=0.0,
    max_epochs=MAX_EPOCHS_FIXED,
    early_stopping=False
)
runs_rows.append({
    "experiment_id": "E3",
    "dataset": DATASET_NAME,
    "seed": SEED,
    "model_summary": model_summary_str(HIDDEN, 0.0, True),
    "optimizer": "Adam",
    "lr": BASE_LR,
    "momentum": 0.0,
    "weight_decay": 0.0,
    "epochs_trained": E3.epochs_trained,
    "best_val_accuracy": E3.best_val_acc,
    "best_val_loss": E3.best_val_loss
})

# E4: EarlyStopping on best of (E2/E3)
best_base = E2 if E2.best_val_acc >= E3.best_val_acc else E3
best_is_dropout = (best_base.experiment_id == "E2")
best_is_bn = (best_base.experiment_id == "E3")

E4_MAX_EPOCHS = 50
ES_PATIENCE = 4

E4 = run_experiment(
    experiment_id="E4",
    hidden_sizes=HIDDEN,
    dropout_p=(DROPOUT_P if best_is_dropout else 0.0),
    use_batchnorm=best_is_bn,
    optimizer_name="Adam",
    lr=BASE_LR,
    momentum=0.0,
    weight_decay=0.0,
    max_epochs=E4_MAX_EPOCHS,
    early_stopping=True,
    es_patience=ES_PATIENCE
)

runs_rows.append({
    "experiment_id": "E4",
    "dataset": DATASET_NAME,
    "seed": SEED,
    "model_summary": model_summary_str(HIDDEN, (DROPOUT_P if best_is_dropout else 0.0), best_is_bn),
    "optimizer": "Adam",
    "lr": BASE_LR,
    "momentum": 0.0,
    "weight_decay": 0.0,
    "epochs_trained": E4.epochs_trained,
    "best_val_accuracy": E4.best_val_acc,
    "best_val_loss": E4.best_val_loss
})

print("\nPart A results:")
print("E1 best val acc:", E1.best_val_acc)
print("E2 best val acc:", E2.best_val_acc)
print("E3 best val acc:", E3.best_val_acc)
print("E4 best val acc:", E4.best_val_acc, "epochs_trained:", E4.epochs_trained)


# -----------------------------
# 10) Save best model + config + curves_best + one-time test eval
# -----------------------------
BEST_MODEL_PATH = ART_DIR / "best_model.pt"
BEST_CONFIG_PATH = ART_DIR / "best_config.json"
CURVES_BEST_PATH = FIG_DIR / "curves_best.png"

torch.save(E4.best_state, BEST_MODEL_PATH)

best_config: Dict[str, Any] = {
    "dataset": DATASET_NAME,
    "seed": SEED,
    "input_dim": input_dim,
    "num_classes": num_classes,
    "model": {
        "hidden_sizes": list(HIDDEN),
        "activation": "ReLU",
        "dropout_p": (DROPOUT_P if best_is_dropout else 0.0),
        "use_batchnorm": bool(best_is_bn)
    },
    "training": {
        "loss": "CrossEntropyLoss",
        "optimizer": "Adam",
        "lr": BASE_LR,
        "weight_decay": 0.0,
        "batch_size": BATCH_SIZE,
        "max_epochs": E4_MAX_EPOCHS,
        "early_stopping": {"enabled": True, "patience": ES_PATIENCE, "metric": "val_accuracy"}
    }
}

with open(BEST_CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(best_config, f, ensure_ascii=False, indent=2)

plot_curves(E4.history, title="E4 best run: train/val loss", out_path=CURVES_BEST_PATH)

best_model = MLP(
    input_dim=input_dim,
    num_classes=num_classes,
    hidden_sizes=HIDDEN,
    dropout_p=best_config["model"]["dropout_p"],
    use_batchnorm=best_config["model"]["use_batchnorm"]
).to(device)

best_model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))

criterion = nn.CrossEntropyLoss()
test_loss, test_acc = evaluate(best_model, test_loader, criterion, device)

print("\nFinal TEST accuracy (best E4 model):", test_acc)
print("Saved:", BEST_MODEL_PATH)
print("Saved:", BEST_CONFIG_PATH)
print("Saved:", CURVES_BEST_PATH)


# -----------------------------
# 11) Part B (O1-O3)
# -----------------------------
# O1: LR too big
O1 = run_experiment(
    experiment_id="O1",
    hidden_sizes=HIDDEN,
    dropout_p=best_config["model"]["dropout_p"],
    use_batchnorm=best_config["model"]["use_batchnorm"],
    optimizer_name="Adam",
    lr=1e-1,
    momentum=0.0,
    weight_decay=0.0,
    max_epochs=6,
    early_stopping=False
)
runs_rows.append({
    "experiment_id": "O1",
    "dataset": DATASET_NAME,
    "seed": SEED,
    "model_summary": model_summary_str(HIDDEN, best_config["model"]["dropout_p"], best_config["model"]["use_batchnorm"]),
    "optimizer": "Adam",
    "lr": 1e-1,
    "momentum": 0.0,
    "weight_decay": 0.0,
    "epochs_trained": O1.epochs_trained,
    "best_val_accuracy": O1.best_val_acc,
    "best_val_loss": O1.best_val_loss
})

# O2: LR too small
O2 = run_experiment(
    experiment_id="O2",
    hidden_sizes=HIDDEN,
    dropout_p=best_config["model"]["dropout_p"],
    use_batchnorm=best_config["model"]["use_batchnorm"],
    optimizer_name="Adam",
    lr=1e-5,
    momentum=0.0,
    weight_decay=0.0,
    max_epochs=6,
    early_stopping=False
)
runs_rows.append({
    "experiment_id": "O2",
    "dataset": DATASET_NAME,
    "seed": SEED,
    "model_summary": model_summary_str(HIDDEN, best_config["model"]["dropout_p"], best_config["model"]["use_batchnorm"]),
    "optimizer": "Adam",
    "lr": 1e-5,
    "momentum": 0.0,
    "weight_decay": 0.0,
    "epochs_trained": O2.epochs_trained,
    "best_val_accuracy": O2.best_val_acc,
    "best_val_loss": O2.best_val_loss
})

# O3: SGD + momentum + weight decay
O3 = run_experiment(
    experiment_id="O3",
    hidden_sizes=HIDDEN,
    dropout_p=best_config["model"]["dropout_p"],
    use_batchnorm=best_config["model"]["use_batchnorm"],
    optimizer_name="SGD",
    lr=1e-2,
    momentum=0.9,
    weight_decay=1e-4,
    max_epochs=12,
    early_stopping=False
)
runs_rows.append({
    "experiment_id": "O3",
    "dataset": DATASET_NAME,
    "seed": SEED,
    "model_summary": model_summary_str(HIDDEN, best_config["model"]["dropout_p"], best_config["model"]["use_batchnorm"]),
    "optimizer": "SGD",
    "lr": 1e-2,
    "momentum": 0.9,
    "weight_decay": 1e-4,
    "epochs_trained": O3.epochs_trained,
    "best_val_accuracy": O3.best_val_acc,
    "best_val_loss": O3.best_val_loss
})

CURVES_LR_PATH = FIG_DIR / "curves_lr_extremes.png"
plot_lr_extremes(O1.history, O2.history, out_path=CURVES_LR_PATH)
print("\nSaved:", CURVES_LR_PATH)


# -----------------------------
# 12) Save runs.csv + summary
# -----------------------------
write_runs_csv(runs_rows, RUNS_CSV_PATH)

print("\nSaved runs:", RUNS_CSV_PATH)
print("\n=== QUICK SUMMARY FOR report.md ===")
bestA = max([r for r in runs_rows if r["experiment_id"] in ["E1", "E2", "E3", "E4"]],
            key=lambda x: float(x["best_val_accuracy"]))
print("Best in Part A:", bestA["experiment_id"], "best_val_accuracy:", bestA["best_val_accuracy"])
print("Final test_accuracy (E4 best model):", test_acc)
print("Artifacts:")
print(" - runs.csv:", RUNS_CSV_PATH)
print(" - best_model.pt:", BEST_MODEL_PATH)
print(" - best_config.json:", BEST_CONFIG_PATH)
print(" - curves_best.png:", CURVES_BEST_PATH)
print(" - curves_lr_extremes.png:", CURVES_LR_PATH)

Device: cpu
REPO_ROOT: C:\Users\User\Desktop\GitHub Cafedra\Cafedra-AI-MALOFEEV-ALEXSANDR
HW_DIR: C:\Users\User\Desktop\GitHub Cafedra\Cafedra-AI-MALOFEEV-ALEXSANDR\homeworks\HW08-09
ART_DIR: C:\Users\User\Desktop\GitHub Cafedra\Cafedra-AI-MALOFEEV-ALEXSANDR\homeworks\HW08-09\artifacts
FIG_DIR: C:\Users\User\Desktop\GitHub Cafedra\Cafedra-AI-MALOFEEV-ALEXSANDR\homeworks\HW08-09\artifacts\figures
DATA_DIR: C:\Users\User\Desktop\GitHub Cafedra\Cafedra-AI-MALOFEEV-ALEXSANDR\data
Sizes: 40000 10000 10000
Sanity batch x: torch.Size([128, 3, 32, 32]) torch.float32 min/max: -1.0 1.0
Sanity batch y: torch.Size([128]) torch.int64 min/max: 0 9
input_dim: 3072 num_classes: 10

Part A results:
E1 best val acc: 0.5346123417721519
E2 best val acc: 0.5435126582278481
E3 best val acc: 0.5535996835443038
E4 best val acc: 0.5573575949367089 epochs_trained: 12

Final TEST accuracy (best E4 model): 0.5510284810126582
Saved: C:\Users\User\Desktop\GitHub Cafedra\Cafedra-AI-MALOFEEV-ALEXSANDR\homeworks\HW08-